# 04 Model: baselines and LASSO

Drafted with an AI assistant, see docs/ai_log.md.

Input: `data/processed/model_table.parquet`.
Outputs: `data/processed/predictions.parquet` (`egid, cutoff, split, model, score`) and `figures/table_lasso_coef.tex` (written here because it needs the fitted model).

Three models; each adds one frame to `frames`:
- `roof_yield` (baseline, nothing fitted): yearly electricity yield of the suitable roof surfaces.
- `nbr_share` (baseline, nothing fitted): share of PV buildings among the neighbours within 250 m (`nbr_pv_share_250`; the radius was fixed before any result, see docs/decisions.md).
- `lasso`: L1-penalised logistic regression, fitted on the train split only.

In [ ]:
import time

import helpers
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

In [ ]:
model_table = pd.read_parquet(helpers.PROCESSED / "model_table.parquet")
keys = model_table[["egid", "cutoff", "split"]]
frames = []

# Baseline 1: roof yield
frames.append(keys.assign(model="roof_yield", score=model_table["yield_kwh"]))

# Baseline 2: neighbour PV share at 250 m
frames.append(keys.assign(model="nbr_share", score=model_table["nbr_pv_share_250"]))

## LASSO

Everything fitted comes from the train split: decile edges, medians, category levels, means and standard deviations (`helpers.fit_features`), the penalty `C` and the coefficients. The test split is only scored.

Penalty `C` is chosen on a log grid of 10 values by 5-fold cross-validation grouped by municipality (`ggdenr`, `helpers.grouped_cv`), scoring ROC AUC, class weights `balanced`. The best mean AUC decides. The one-standard-error value (smallest `C`, i.e. strongest penalty, within one standard error of the best) is printed for information only. The standardisation is fitted once on the whole train split, not inside each fold (a small, usual shortcut).

In [ ]:
train = model_table[model_table["split"] == "train"].reset_index(drop=True)
test = model_table[model_table["split"] == "test"].reset_index(drop=True)

# Features: fit on train, apply to both
params = helpers.fit_features(train)
x_train = helpers.apply_features(train, params)
x_test = helpers.apply_features(test, params)
y_train = train["y"].to_numpy()
print(x_train.shape[1], "features")


def fit_lasso(x, y, c):
    # l1_ratio=1 is the pure L1 penalty (the `penalty` argument is deprecated)
    model = LogisticRegression(
        l1_ratio=1, C=c, solver="liblinear", class_weight="balanced",
        tol=1e-5, max_iter=1000, random_state=0,
    )  # fmt: skip
    return model.fit(x, y)


# Grouped 5-fold CV over the grid of C (the grid reaches the plateau of the CV curve)
start = time.time()
c_grid = np.logspace(-3, -1, 10)
folds = helpers.grouped_cv(train["ggdenr"], n_splits=5)
cv_rows = []
for c_value in c_grid:
    aucs, n_nonzero = [], []
    for fit_idx, val_idx in folds:
        model = fit_lasso(x_train.iloc[fit_idx], y_train[fit_idx], c_value)
        aucs.append(roc_auc_score(y_train[val_idx], model.decision_function(x_train.iloc[val_idx])))
        n_nonzero.append(int((model.coef_ != 0).sum()))
    cv_rows.append(
        {
            "C": c_value,
            "mean_auc": np.mean(aucs),
            "se_auc": np.std(aucs, ddof=1) / np.sqrt(len(aucs)),
            "mean_nonzero": np.mean(n_nonzero),
        }
    )
cv = pd.DataFrame(cv_rows)
print(cv.round(5).to_string(index=False))

# Best mean AUC is the model used; one-standard-error C is for information only
best = cv["mean_auc"].idxmax()
c_best = cv.loc[best, "C"]
within = cv[cv["mean_auc"] >= cv.loc[best, "mean_auc"] - cv.loc[best, "se_auc"]]
c_1se = within["C"].min()

# Refit on the whole train split
lasso = fit_lasso(x_train, y_train, c_best)
n_nonzero = int((lasso.coef_ != 0).sum())
n_nonzero_1se = int((fit_lasso(x_train, y_train, c_1se).coef_ != 0).sum())
print(f"chosen C {c_best:.5f}: {n_nonzero} of {x_train.shape[1]} coefficients non-zero")
print(f"one-standard-error C {c_1se:.5f}: {n_nonzero_1se} non-zero (information only)")
print(f"fit time {time.time() - start:.0f} s")

In [ ]:
# Coefficient table: non-zero terms on standardised features, largest absolute value first
coef = pd.Series(lasso.coef_[0], index=x_train.columns)
coef = coef[coef != 0].sort_values(key=abs, ascending=False)
lines = [
    r"\begin{tabular}{lr}",
    r"\toprule",
    r"Feature (standardised) & Coefficient \\",
    r"\midrule",
]
for name, value in coef.items():
    lines.append(f"{name.replace('_', chr(92) + '_')} & {value:.3f} \\\\")
lines += [
    r"\midrule",
    r"\multicolumn{2}{l}{"
    + f"$C = {c_best:.4f}$, {n_nonzero} of {x_train.shape[1]} coefficients non-zero, "
    + f"mean CV AUC {cv.loc[best, 'mean_auc']:.4f}"
    + r"} \\",
    r"\bottomrule",
    r"\end{tabular}",
]
(helpers.FIGURES / "table_lasso_coef.tex").write_text("\n".join(lines) + "\n")
print(coef.head(15).round(3).to_string())

Scores are predicted probabilities (with `balanced` class weights they are not calibrated: they over-state the base rate, which is fine for ranking). The `lasso` scores on the train split are in-sample (the model was fitted on those rows); only the test split is out-of-time.

In [ ]:
# LASSO scores: predicted probability, same row order as model_table
score = np.empty(len(model_table))
for split_name, part, x_part in [("train", train, x_train), ("test", test, x_test)]:
    rows = (model_table["split"] == split_name).to_numpy()
    assert (model_table.loc[rows, "egid"].to_numpy() == part["egid"].to_numpy()).all()
    score[rows] = lasso.predict_proba(x_part)[:, 1]
frames.append(keys.assign(model="lasso", score=score))

In [ ]:
# One table, sorted deterministically; scores as float64 rounded to 6 decimals
predictions = pd.concat(frames, ignore_index=True)
predictions["score"] = predictions["score"].astype("float64").round(6)
predictions = predictions.sort_values(["cutoff", "model", "egid"]).reset_index(drop=True)
print(predictions["score"].isna().sum(), "missing scores")
print(predictions.groupby(["model", "split"]).size())
predictions.to_parquet(helpers.PROCESSED / "predictions.parquet", index=False)